In [1]:
# install the python client library for Redis
!pip install redis -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 9.0 MB/s eta 0:00:00


In [2]:
# Redis is not installed by default on a Colab VM, so this installs one
# directly inside the Colab session, same idea as the rabbitmq notebook.
# if you already have the dev Redis url/creds, you can skip this cell and
# just fill in the config cell below instead.
!apt-get install -y -qq redis-server > /dev/null

# start it as a background daemon process
!redis-server --daemonize yes

# quick check that it actually came up
!redis-cli ping

PONG


In [3]:
# ---- Redis connection config ----
# these are the dev Redis connection details. replace host/port/password/db
# with what was given for the dev environment once available.
# use_tls should be True if the dev Redis instance requires a TLS connection
# (rediss:// instead of redis://).
'''
REDIS_HOST = 'localhost'     # dev Redis hostname goes here
REDIS_PORT = 6379            # dev Redis port goes here
REDIS_PASSWORD = None        # dev Redis password goes here, None if no auth
REDIS_DB = 0                 # dev Redis db number goes here
REDIS_USE_TLS = False        # set True if the dev instance requires TLS
'''

REDIS_HOST = '129.153.75.221'
REDIS_PORT = 6379
REDIS_USERNAME = 'default'
REDIS_PASSWORD = 'cwe5cU6Tyzvd'
REDIS_DB = 0                 # not given, 0 is the standard default db
REDIS_USE_TLS = False        # not stated either way, leaving off unless told otherwise


In [4]:
import redis

# connect to redis using the config above.
# decode_responses=True so values come back as normal python strings instead
# of bytes, which is easier to work with when we're storing json.
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    ssl=REDIS_USE_TLS,
    decode_responses=True
)

# ping is the standard way to confirm the connection actually works
print('connected to redis:', redis_client.ping())

connected to redis: True


In [5]:
import json

# ---- write / set ----
# cache a sample prediction result under a key. using a short ttl (expiry) of
# 1 hour so stale predictions do not sit in the cache forever - after the ttl
# passes redis just drops the key on its own.

cache_key = 'yield:demo_applicant_001'

sample_result = {
    'yield_probability': 0.62,
    'predicted_enrolled': True
}

redis_client.set(cache_key, json.dumps(sample_result), ex=3600)
print('set key:', cache_key, '->', sample_result)

set key: yield:demo_applicant_001 -> {'yield_probability': 0.62, 'predicted_enrolled': True}


In [6]:
# ---- read / get ----
# read the value back out and parse it from json back into a python dict

cached_value = redis_client.get(cache_key)

if cached_value:
    result = json.loads(cached_value)
    print('got value for', cache_key, ':', result)
else:
    print('nothing cached under', cache_key)

got value for yield:demo_applicant_001 : {'yield_probability': 0.62, 'predicted_enrolled': True}


In [7]:
# ---- update ----
# updating a cached value in redis is just setting the same key again with
# the new value. here we simulate the applicant getting re-scored with a
# different aid package and the cached prediction changing as a result.

updated_result = {
    'yield_probability': 0.71,
    'predicted_enrolled': True
}

redis_client.set(cache_key, json.dumps(updated_result), ex=3600)

# read it back to confirm the update actually took
print('updated value:', json.loads(redis_client.get(cache_key)))

updated value: {'yield_probability': 0.71, 'predicted_enrolled': True}


In [8]:
# ---- delete ----
# remove the cached entry. delete() returns the number of keys that were
# actually removed (0 if the key did not exist).

deleted_count = redis_client.delete(cache_key)
print('keys deleted:', deleted_count)

# confirm it is actually gone
print('value after delete:', redis_client.get(cache_key))


keys deleted: 1
value after delete: None
